In [13]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType
from src.spark_session import get_spark
from src.config import ORDERS_RAW_PATH, ORDERS_SILVER_PATH


In [14]:
spark = get_spark("SilverOrders")

In [15]:
orders_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(ORDERS_RAW_PATH))
)

orders_raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



In [16]:
print("Rows: ", orders_raw_df.count())
print("Columns: ", len(orders_raw_df.columns))
print("Column names: ", orders_raw_df.columns)

Rows:  99441
Columns:  8
Column names:  ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [17]:
rows_count = orders_raw_df.count()

distinct_orders_count = orders_raw_df.select("order_id").distinct().count()

print("Rows count: ", rows_count)
print("Distinct order IDs: ", distinct_orders_count)
print("order_id is unique: ", distinct_orders_count == rows_count)

Rows count:  99441
Distinct order IDs:  99441
order_id is unique:  True


In [18]:
orders_status_counts = orders_raw_df.groupBy('order_status').count().orderBy(F.col("count"), ascending=False)
orders_status_counts.show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [19]:
delivered_without_delivery_dates = (
    orders_raw_df
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("order_delivered_customer_date").isNull())
)

delivered_without_delivery_dates.show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|ec05a6d8558c6455f0cbbd8a420ad34f|delivered   |2017-11-28 17:44:07     |2017-11-28 17:56:40|2017-11-30 18:12:23         |NULL                         |2017-12-18 00:00:00          |
|f5dd62b788049ad9fc0526e3ad11a097|5e89028e024b381dc84a13a3570decb4|delivered   |2018-06-20 06:58:43     |2018-06-20 07:19:05|2018-06-25 08:0

In [20]:
not_delivered_with_delivery_date = (
    orders_raw_df
    .filter((F.col("order_status") != "delivered") & (F.col("order_delivered_customer_date").isNotNull()))
)

not_delivered_with_delivery_date.show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|1950d777989f6a877539f53795b4c3c3|1bccb206de9f0f25adc6871a1bcf77b2|canceled    |2018-02-19 19:48:52     |2018-02-19 20:56:05|2018-02-20 19:57:13         |2018-03-21 22:03:51          |2018-03-09 00:00:00          |
|dabf2b0e35b423f94618bf965fcb7514|5cdec0bb8cbdf53ffc8fdc212cd247c6|canceled    |2016-10-09 00:56:52     |2016-10-09 13:36:58|2016-10-13 13:3

In [21]:
print("Delivered without delivery date: ", delivered_without_delivery_dates.count())
print("Not delivered with delivery date: ", not_delivered_with_delivery_date.count())

Delivered without delivery date:  8
Not delivered with delivery date:  6


In [22]:
orders_with_quality = orders_raw_df.withColumn(
    "delivery_status_quality",
    F.when(
        (F.col("order_status") == "delivered") & (F.col("order_delivered_customer_date").isNull()),
        F.lit("DELIVERED_WITHOUT_DATE")
    )
    .when(
        ((F.col("order_status") != "delivered") & (F.col("order_delivered_customer_date").isNotNull())),
        F.lit("DATE_WITHOUT_DELIVERED_STATUS")
    ).otherwise(F.lit("VALID"))

)

orders_with_quality.groupBy('delivery_status_quality').count().show(truncate=False)

+-----------------------------+-----+
|delivery_status_quality      |count|
+-----------------------------+-----+
|VALID                        |99427|
|DATE_WITHOUT_DELIVERED_STATUS|6    |
|DELIVERED_WITHOUT_DATE       |8    |
+-----------------------------+-----+



In [23]:
timestamp_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders_typed = orders_with_quality

for col_name in timestamp_columns:
    orders_typed = orders_typed.withColumn(col_name, F.to_timestamp(F.col(col_name)))

orders_typed.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_status_quality: string (nullable = false)



In [24]:
orders_typed.select(
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
).show(5, truncate=False)

+--------------------------------+------------------------+-----------------------------+-----------------------------+
|order_id                        |order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|2017-10-02 10:56:33     |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|2018-07-24 20:41:37     |2018-08-07 15:27:45          |2018-08-13 00:00:00          |
|47770eb9100c2d0c44946d9cf07ec65d|2018-08-08 08:38:49     |2018-08-17 18:06:29          |2018-09-04 00:00:00          |
|949d5b44dbf5de918fe9c16f97b45f8a|2017-11-18 19:28:06     |2017-12-02 00:28:42          |2017-12-15 00:00:00          |
|ad21c59c0840e6cb83a9ceb5573f8159|2018-02-13 21:18:39     |2018-02-16 18:17:02          |2018-02-26 00:00:00          |
+--------------------------------+------

In [25]:
orders_metrics = (
    orders_typed
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delays_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
)

In [26]:
orders_metrics.select(
    'order_id',
    'order_status',
    'delivery_days',
    'delivery_delays_days',
).show(10, truncate=False)

+--------------------------------+------------+-------------+--------------------+
|order_id                        |order_status|delivery_days|delivery_delays_days|
+--------------------------------+------------+-------------+--------------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |8            |-8                  |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |14           |-6                  |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |9            |-18                 |
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |14           |-13                 |
|ad21c59c0840e6cb83a9ceb5573f8159|delivered   |3            |-10                 |
|a4591c265e18cb1dcee52889e2d8acc3|delivered   |17           |-6                  |
|136cce7faa42fdb2cefd53fdc79a6098|invoiced    |NULL         |NULL                |
|6514b8ad8028c9f2cc2374ded245783f|delivered   |10           |-12                 |
|76c6e866289321a7c93b82b54852dc33|delivered   |10           |-32                 |
|e69

In [27]:
delayed_orders = orders_metrics.filter(F.col('delivery_delays_days') > 0).count()
print(delayed_orders)

6535


In [28]:
delivered_orders = orders_metrics.filter(F.col("order_status") == "delivered")
late_delivered_orders = orders_metrics.filter(F.col('delivery_delays_days') > 0)

delivered_count = delivered_orders.count()
late_count = late_delivered_orders.count()

late_percentage = late_count / delivered_count * 100

print("Delivered orders: ", delivered_count)
print("Late delivered orders: ", late_count)
print(f"Late delivery percentage: {round(late_percentage, 2)} %")

Delivered orders:  96478
Late delivered orders:  6535
Late delivery percentage: 6.77 %


In [29]:
orders_metrics.filter(
    F.col("delivery_delays_days") > 0
).select(
    F.min("delivery_delays_days").alias("min_delay"),
    F.avg("delivery_delays_days").alias("avg_delay"),
    F.max("delivery_delays_days").alias("max_delay"),
).show()

+---------+----------------+---------+
|min_delay|       avg_delay|max_delay|
+---------+----------------+---------+
|        1|10.6203519510329|      188|
+---------+----------------+---------+



In [30]:
orders_metrics.filter(
    F.col("delivery_delays_days") > 0
).select(
    'order_id',
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delays_days"
).orderBy(
    F.col("delivery_delays_days").desc(),
).show(10, truncate=False)

+--------------------------------+------------------------+-----------------------------+-----------------------------+--------------------+
|order_id                        |order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|delivery_delays_days|
+--------------------------------+------------------------+-----------------------------+-----------------------------+--------------------+
|1b3190b2dfa9d789e1f14c05b647a14a|2018-02-23 14:57:35     |2018-09-19 23:24:07          |2018-03-15 00:00:00          |188                 |
|ca07593549f1816d26a572e06dc1eab6|2017-02-21 23:31:27     |2017-09-19 14:36:39          |2017-03-22 00:00:00          |181                 |
|47b40429ed8cce3aee9199792275433f|2018-01-03 09:44:01     |2018-07-13 20:51:31          |2018-01-19 00:00:00          |175                 |
|2fe324febf907e3ea3f2aa9650869fa5|2017-03-13 20:17:10     |2017-09-19 17:00:07          |2017-04-05 00:00:00          |167                 |
|285ab9426d69

In [31]:
orders_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("customer_id", StringType(), nullable=False),
    StructField("order_status", StringType(), nullable=True),
    StructField("order_purchase_timestamp", TimestampType(), nullable=True),
    StructField("order_approved_at", TimestampType(), nullable=True),
    StructField("order_delivered_carrier_date", TimestampType(), nullable=True),
    StructField("order_delivered_customer_date", TimestampType(), nullable=True),
    StructField("order_estimated_delivery_date", TimestampType(), nullable=True),
])

In [32]:
orders_typed_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(str(ORDERS_RAW_PATH))
)

orders_typed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [33]:
orders_typed_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [34]:
print("Rows: ", orders_typed_df.count())
print("Columns: ", len(orders_typed_df.columns))

Rows:  99441
Columns:  8


In [35]:
orders_typed_df.select([
    F.sum(F.col(col_name).isNull().cast("int")).alias(col_name)
    for col_name in orders_typed_df.columns
]).show(truncate=False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [36]:
clean_orders_df = (
    orders_typed_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
    .dropDuplicates(["order_id"])
)

In [37]:
clean_orders_df.count()

99441

In [39]:
allowed_statuses = [
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "unavailable",
    "canceled"
]

invalid_status_orders = clean_orders_df.filter(
    ~F.col("order_status").isin(allowed_statuses)
)

In [40]:
invalid_status_orders.groupBy("order_status").count().show()

+------------+-----+
|order_status|count|
+------------+-----+
+------------+-----+



In [41]:
clean_orders_df = (
    clean_orders_df
    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
    .withColumn(
        "is_late",
        F.when(
            F.col("delivery_delay_days") > 0,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

In [42]:
clean_orders_df.show(20, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|00018f77f2f0320c557190d7a144bdd3|f6dd3ec061db4e3987629fe6b26e5cce|delivered   |2017-04-26 10:53:06     |2017-04-26 11:05:13|2017-05-04 14:35:00         |2017-05-12 16:04:24          |2017-05-15 00:00:00          |16           |-

In [43]:
clean_orders_df = clean_orders_df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "delivery_delay_days",
    "is_late"
)

In [44]:
clean_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = false)



In [45]:
print("Clean row count:", clean_orders_df.count())
print("Distinct order IDs:", clean_orders_df.select("order_id").distinct().count())

Clean row count: 99441
Distinct order IDs: 99441


In [46]:
clean_orders_df.filter(
    F.col("delivery_days") < 0
).show(5, truncate=False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+



In [47]:
clean_orders_df.write.mode("overwrite").parquet(str(ORDERS_SILVER_PATH))

In [48]:
saved_orders_df = spark.read.parquet(str(ORDERS_SILVER_PATH))
saved_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)

